# Batch Image Upscaler (Akselerasi GPU T4 - Turbo Mode)

Aplikasi batch upscaler gambar dengan arsitektur Real-ESRGAN dan akselerasi GPU Nvidia T4.
- **Mode Turbo GPU**: Tile 1024 memanfaatkan VRAM 10-12 GB untuk kecepatan maksimal.
- **Pilihan Skala**: 2x (resolusi pas dan tajam) atau 4x (resolusi maksimal).
- **Format Output**: JPEG Kualitas Tinggi (Quality 96, Subsampling 4:4:4).
- **Anti-OOM**: Tiling adaptif dan pembersihan cache VRAM otomatis.

---  
**Pastikan Runtime GPU Aktif**: Menu `Runtime` -> `Change runtime type` -> pilih `T4 GPU` -> `Save`.

In [ ]:
# 1. Verifikasi Status GPU
!nvidia-smi

In [ ]:
# 2. Inisialisasi Model & Engine Real-ESRGAN (Turbo cuDNN)
import os, glob, math, shutil, sys
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

# Optimasi performa cuDNN untuk GPU Nvidia T4
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

os.makedirs("weights", exist_ok=True)
os.makedirs("inputs", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print("Memeriksa model RealESRGAN_x4plus.pth...")
!wget -c -q --show-progress "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth" -P weights/

# Arsitektur Neural Network (RRDBNet Pure PyTorch)
class ResidualDenseBlock(nn.Module):
    def __init__(self, num_feat=64, num_grow_ch=32):
        super().__init__()
        self.conv1 = nn.Conv2d(num_feat, num_grow_ch, 3, 1, 1)
        self.conv2 = nn.Conv2d(num_feat + num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv3 = nn.Conv2d(num_feat + 2 * num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv4 = nn.Conv2d(num_feat + 3 * num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv5 = nn.Conv2d(num_feat + 4 * num_grow_ch, num_feat, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)
    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        return x5 * 0.2 + x

class RRDB(nn.Module):
    def __init__(self, num_feat=64, num_grow_ch=32):
        super().__init__()
        self.rdb1 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb2 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb3 = ResidualDenseBlock(num_feat, num_grow_ch)
    def forward(self, x):
        out = self.rdb1(x)
        out = self.rdb2(out)
        out = self.rdb3(out)
        return out * 0.2 + x

class RRDBNet(nn.Module):
    def __init__(self, num_in_ch=3, num_out_ch=3, scale=4, num_feat=64, num_block=23, num_grow_ch=32):
        super().__init__()
        self.scale = scale
        self.conv_first = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)
        self.body = nn.Sequential(*[RRDB(num_feat, num_grow_ch) for _ in range(num_block)])
        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up2 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_hr = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)
    def forward(self, x):
        feat = self.conv_first(x)
        body_feat = self.conv_body(self.body(feat))
        feat = feat + body_feat
        feat = self.lrelu(self.conv_up1(F.interpolate(feat, scale_factor=2, mode="nearest")))
        feat = self.lrelu(self.conv_up2(F.interpolate(feat, scale_factor=2, mode="nearest")))
        out = self.conv_last(self.lrelu(self.conv_hr(feat)))
        return out

def load_model(model_path="weights/RealESRGAN_x4plus.pth", device=None, fp16=True):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model tidak ada di: {model_path}")
    model = RRDBNet(num_in_ch=3, num_out_ch=3, scale=4, num_feat=64, num_block=23, num_grow_ch=32)
    try:
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    except TypeError:
        checkpoint = torch.load(model_path, map_location=device)
    state_dict = checkpoint.get("params_ema", checkpoint.get("params", checkpoint))
    model.load_state_dict(state_dict, strict=True)
    model.eval().to(device)
    return model.half() if (fp16 and device.type == "cuda") else model

def process_tiles(img_tensor, model, tile=1024, tile_pad=10, scale=4):
    batch, channel, height, width = img_tensor.shape
    if tile == 0 or (height <= tile and width <= tile):
        with torch.inference_mode():
            return model(img_tensor)
    out_tensor = torch.zeros((batch, channel, height * scale, width * scale), dtype=img_tensor.dtype, device=img_tensor.device)
    tiles_x = math.ceil(width / tile)
    tiles_y = math.ceil(height / tile)
    for y in range(tiles_y):
        for x in range(tiles_x):
            in_x = x * tile
            in_y = y * tile
            in_x_end = min(in_x + tile, width)
            in_y_end = min(in_y + tile, height)
            in_x_pad = max(in_x - tile_pad, 0)
            in_x_end_pad = min(in_x_end + tile_pad, width)
            in_y_pad = max(in_y - tile_pad, 0)
            in_y_end_pad = min(in_y_end + tile_pad, height)
            tile_in = img_tensor[:, :, in_y_pad:in_y_end_pad, in_x_pad:in_x_end_pad]
            with torch.inference_mode():
                tile_out = model(tile_in)
            out_x = in_x * scale
            out_x_end = in_x_end * scale
            out_y = in_y * scale
            out_y_end = in_y_end * scale
            t_out_x = (in_x - in_x_pad) * scale
            t_out_x_end = t_out_x + (in_x_end - in_x) * scale
            t_out_y = (in_y - in_y_pad) * scale
            t_out_y_end = t_out_y + (in_y_end - in_y) * scale
            out_tensor[:, :, out_y:out_y_end, out_x:out_x_end] = tile_out[:, :, t_out_y:t_out_y_end, t_out_x:t_out_x_end]
    return out_tensor

def upscale_image(pil_img, model, device, tile=1024, fp16=True, outscale=4.0):
    orig_w, orig_h = pil_img.size
    img = pil_img.convert("RGB")
    np_img = np.array(img, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(np_img).permute(2, 0, 1).unsqueeze(0).to(device)
    if fp16 and device.type == "cuda":
        tensor = tensor.half()
    out_tensor = process_tiles(tensor, model, tile=tile, tile_pad=10, scale=4)
    out_tensor = out_tensor.squeeze(0).float().clamp(0.0, 1.0)
    out_np = (out_tensor.permute(1, 2, 0).cpu().numpy() * 255.0).round().astype(np.uint8)
    res_img = Image.fromarray(out_np)
    if outscale != 4.0:
        res_img = res_img.resize((int(round(orig_w * outscale)), int(round(orig_h * outscale))), Image.Resampling.LANCZOS)
    return res_img

def get_image_files(input_dir="inputs"):
    os.makedirs(input_dir, exist_ok=True)
    valid_exts = (".png", ".jpg", ".jpeg", ".webp", ".bmp")
    if os.path.exists("/content"):
        try:
            for fname in os.listdir("/content"):
                fpath = os.path.join("/content", fname)
                if os.path.isfile(fpath) and fname.lower().endswith(valid_exts):
                    dest = os.path.join(input_dir, fname)
                    if not os.path.exists(dest):
                        shutil.move(fpath, dest)
        except Exception:
            pass
    files = []
    if os.path.exists(input_dir):
        for fname in os.listdir(input_dir):
            fpath = os.path.join(input_dir, fname)
            if os.path.isfile(fpath) and fname.lower().endswith(valid_exts):
                files.append(fpath)
    return sorted(files)

def clean_folders(input_dir="inputs", output_dir="outputs"):
    for folder in [input_dir, output_dir]:
        if os.path.exists(folder):
            for f in os.listdir(folder):
                p = os.path.join(folder, f)
                if os.path.isfile(p): os.remove(p)
                elif os.path.isdir(p): shutil.rmtree(p)
    if os.path.exists("hasil_upscale.zip"): os.remove("hasil_upscale.zip")

print("Engine Real-ESRGAN Turbo Mode siap digunakan.")

In [ ]:
# 3. Unggah File Gambar Bahan
# Klik tombol 'Choose Files' di bawah, lalu pilih semua gambar yang ingin di-upscale:
import os, shutil
from google.colab import files

os.makedirs("inputs", exist_ok=True)
print("Silakan pilih file gambar bahan:")
uploaded = files.upload()
for fn in uploaded.keys():
    dest = os.path.join("inputs", fn)
    if os.path.exists(dest):
        os.remove(dest)
    shutil.move(fn, dest)

print(f"\nSUKSES: {len(uploaded)} file berhasil dimasukkan ke folder inputs/.")
print("Silakan lanjutkan ke Cell 4 di bawah untuk menjalankan proses upscale.")

In [ ]:
# 4. PANEL KONTROL EKSEKUSI & UNDUH (WINDOWS THEME - TURBO)
import os, glob, shutil
from PIL import Image
import torch
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import files

# --- LANGKAH 1: STATUS FILE ---
lbl_file_status = widgets.HTML(value="")
btn_check_files = widgets.Button(
    description='Periksa Ulang Folder inputs',
    layout=widgets.Layout(width='220px', height='36px')
)

def refresh_file_status():
    fl = get_image_files("inputs")
    if fl:
        names_preview = ", ".join([os.path.basename(f) for f in fl[:3]])
        if len(fl) > 3:
            names_preview += f" ... (+{len(fl)-3} file lainnya)"
        lbl_file_status.value = f"<div style='color:#0078D4; font-weight:600; font-size:13px; margin:4px 0;'>STATUS: {len(fl)} file siap diproses.<br><span style='font-size:12px; font-weight:normal; color:#605E5C;'>File: {names_preview}</span></div>"
    else:
        lbl_file_status.value = "<div style='color:#C42B1C; font-weight:600; font-size:13px; margin:4px 0;'>STATUS: 0 file terdeteksi. Jalankan Cell 3 di atas untuk mengunggah gambar.</div>"

btn_check_files.on_click(lambda b: refresh_file_status())
refresh_file_status()

# --- LANGKAH 2: PARAMETER ---
perf_choice = widgets.ToggleButtons(
    options=[('Turbo (Tile 1024 - Maksimal GPU & Cepat)', 1024), ('Standar (Tile 512 - Hemat VRAM)', 512)],
    value=1024,
    description='Performa GPU:',
    layout=widgets.Layout(margin='6px 0')
)

scale_choice = widgets.ToggleButtons(
    options=[('2x (Rekomendasi - Tajam & Pas)', 2.0), ('4x (Maksimal Resolusi)', 4.0)],
    value=2.0,
    description='Skala:',
    layout=widgets.Layout(margin='6px 0')
)

quality_slider = widgets.IntSlider(
    value=96, min=85, max=100, step=1,
    description='Kualitas JPG:',
    layout=widgets.Layout(width='360px')
)

# --- LANGKAH 3: EKSEKUSI ---
btn_run = widgets.Button(
    description='Mulai Proses Upscale',
    layout=widgets.Layout(width='220px', height='40px')
)
btn_run.style.button_color = '#0078D4'
btn_run.style.text_color = '#FFFFFF'

progress_bar = widgets.IntProgress(
    value=0, min=0, max=100,
    orientation='horizontal',
    layout=widgets.Layout(width='100%', height='22px', display='none', margin='8px 0')
)
progress_bar.style.bar_color = '#0078D4'

lbl_progress = widgets.HTML(value="<div style='color:#605E5C; font-size:13px;'>Menunggu perintah eksekusi.</div>")

# --- LANGKAH 4: UNDUH & RESET ---
alert_box = widgets.HTML(value="")

btn_download = widgets.Button(
    description='Unduh Arsip ZIP',
    layout=widgets.Layout(width='180px', height='38px')
)
btn_download.style.button_color = '#107C10'
btn_download.style.text_color = '#FFFFFF'

btn_clean = widgets.Button(
    description='Bersihkan Folder Batch',
    layout=widgets.Layout(width='200px', height='38px')
)
btn_clean.style.button_color = '#C42B1C'
btn_clean.style.text_color = '#FFFFFF'

def on_run_clicked(b):
    fl = get_image_files("inputs")
    if not fl:
        alert_box.value = "<div style='padding:10px 14px; background:#FDE7E9; border-left:4px solid #C42B1C; color:#A80000; font-weight:600; font-size:13px;'>PERINGATAN: Folder inputs/ kosong. Jalankan Cell 3 terlebih dahulu untuk unggah file.</div>"
        return

    btn_run.disabled = True
    btn_check_files.disabled = True
    btn_download.disabled = True
    btn_clean.disabled = True
    alert_box.value = ""

    progress_bar.layout.display = 'block'
    progress_bar.max = len(fl)
    progress_bar.value = 0
    lbl_progress.value = "<div style='color:#0078D4; font-weight:600; font-size:13px;'>Menginisialisasi model ke GPU (Turbo cuDNN aktif)...</div>"

    s = scale_choice.value
    q = quality_slider.value
    tile_val = perf_choice.value
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = load_model("weights/RealESRGAN_x4plus.pth", device, fp16=True)

    os.makedirs("outputs", exist_ok=True)
    success = 0
    for idx, path in enumerate(fl, 1):
        fn = os.path.basename(path)
        lbl_progress.value = f"<div style='color:#201F1E; font-size:13px;'><b>[{idx}/{len(fl)}] Memproses (Tile {tile_val}):</b> {fn} ...</div>"
        try:
            with Image.open(path) as src:
                out = upscale_image(src, model, device, tile=tile_val, fp16=True, outscale=s)
                out_name = f"{os.path.splitext(fn)[0]}_upscaled_{int(s)}x.jpg"
                out.save(os.path.join("outputs", out_name), format="JPEG", quality=q, subsampling=0, optimize=True)
                w, h = out.size
                success += 1
                lbl_progress.value = f"<div style='color:#107C10; font-size:13px;'><b>[{idx}/{len(fl)}] Selesai:</b> {out_name} ({w}x{h} px)</div>"
        except Exception as err:
            lbl_progress.value = f"<div style='color:#C42B1C; font-size:13px;'><b>[{idx}/{len(fl)}] Gagal:</b> {fn} ({err})</div>"
        finally:
            if device.type == "cuda":
                torch.cuda.empty_cache()
        progress_bar.value = idx

    alert_box.value = f"<div style='padding:12px 16px; background:#DFF6DD; border-left:4px solid #107C10; color:#0E5B0E; font-weight:600; font-size:13px; margin:10px 0;'>STATUS: PROSES SELESAI 100%. Sebanyak {success} dari {len(fl)} gambar berhasil diproses ({int(s)}x). Silakan klik tombol 'Unduh Arsip ZIP'.</div>"
    lbl_progress.value = "<div style='color:#107C10; font-weight:600; font-size:13px;'>Seluruh proses selesai.</div>"

    btn_run.disabled = False
    btn_check_files.disabled = False
    btn_download.disabled = False
    btn_clean.disabled = False

def on_download_clicked(b):
    out_files = glob.glob("outputs/*_upscaled_*.jpg")
    if not out_files:
        alert_box.value = "<div style='padding:10px 14px; background:#FDE7E9; border-left:4px solid #C42B1C; color:#A80000; font-weight:600; font-size:13px;'>PERINGATAN: Folder outputs/ kosong. Jalankan Proses Upscale terlebih dahulu.</div>"
        return
    shutil.make_archive("hasil_upscale", "zip", "outputs")
    size_mb = os.path.getsize("hasil_upscale.zip") / (1024 * 1024)
    alert_box.value = f"<div style='padding:10px 14px; background:#DFF6DD; border-left:4px solid #107C10; color:#0E5B0E; font-weight:600; font-size:13px;'>Mengunduh file hasil_upscale.zip ({size_mb:.2f} MB)...</div>"
    files.download("hasil_upscale.zip")

def on_clean_clicked(b):
    clean_folders("inputs", "outputs")
    refresh_file_status()
    progress_bar.layout.display = 'none'
    progress_bar.value = 0
    lbl_progress.value = "<div style='color:#605E5C; font-size:13px;'>Menunggu perintah eksekusi.</div>"
    alert_box.value = "<div style='padding:10px 14px; background:#EFF6FC; border-left:4px solid #0078D4; color:#004578; font-weight:600; font-size:13px;'>Folder inputs/ dan outputs/ telah dikosongkan. Siap untuk batch baru.</div>"

btn_run.on_click(on_run_clicked)
btn_download.on_click(on_download_clicked)
btn_clean.on_click(on_clean_clicked)

# --- DESAIN STRUKTUR PANEL (WINDOWS THEME) ---
card_css = "background:#FFFFFF; border:1px solid #D2D0CE; border-radius:4px; padding:16px 20px; margin-bottom:12px; box-shadow:0 1.6px 3.6px 0 rgba(0,0,0,0.13), 0 0.3px 0.9px 0 rgba(0,0,0,0.11); font-family:'Segoe UI', system-ui, sans-serif;"

step1 = widgets.VBox([
    widgets.HTML("<div style='font-size:14px; font-weight:600; color:#0078D4; margin-bottom:8px;'>LANGKAH 1: STATUS FILE GAMBAR</div>"),
    lbl_file_status,
    btn_check_files
], layout=widgets.Layout(extra_style=card_css))

step2 = widgets.VBox([
    widgets.HTML("<div style='font-size:14px; font-weight:600; color:#0078D4; margin-bottom:8px;'>LANGKAH 2: PARAMETER KONFIGURASI</div>"),
    perf_choice,
    scale_choice,
    quality_slider
], layout=widgets.Layout(extra_style=card_css))

step3 = widgets.VBox([
    widgets.HTML("<div style='font-size:14px; font-weight:600; color:#0078D4; margin-bottom:8px;'>LANGKAH 3: EKSEKUSI PROSES UPSCALE</div>"),
    btn_run,
    widgets.Box([progress_bar], layout=widgets.Layout(margin='6px 0')),
    lbl_progress
], layout=widgets.Layout(extra_style=card_css))

step4 = widgets.VBox([
    widgets.HTML("<div style='font-size:14px; font-weight:600; color:#0078D4; margin-bottom:8px;'>LANGKAH 4: HASIL UNDUHAN DAN MANAJEMEN BATCH</div>"),
    alert_box,
    widgets.HBox([btn_download, btn_clean], layout=widgets.Layout(margin='6px 0 0 0'))
], layout=widgets.Layout(extra_style=card_css))

app_container = widgets.VBox([
    widgets.HTML("<div style='font-family:\'Segoe UI\', sans-serif; margin-bottom:14px;'><h2 style='color:#201F1E; margin:0;'>BATCH IMAGE UPSCALER (TURBO MODE)</h2><div style='color:#605E5C; font-size:13px; margin-top:4px;'>Eksekusi GPU Colab Nvidia T4 (Real-ESRGAN Pure PyTorch)</div></div>"),
    step1, step2, step3, step4
], layout=widgets.Layout(width='100%', max_width='860px'))

display(app_container)